# AI-Powered Customer Churn Intelligence System

## Day 4 - Machine Learning Modeling & Baseline

This notebook develops a machine learning pipeline to predict customer churn probability.

The workflow includes:

- Feature selection
- Target definition
- Train-test splitting
- Categorical feature encoding
- Numerical feature scaling
- Logistic Regression baseline
- Model evaluation
- Churn probability generation

The model is trained only on customer attributes available before the churn outcome.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [2]:
# Load processed dataset

data_path = "../data/customer_churn_processed.csv"

df = pd.read_csv(data_path)

print("Processed dataset loaded successfully.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Processed dataset loaded successfully.
Shape: (2800, 15)

Columns:
['user_id', 'signup_date', 'plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago', 'churn', 'churn_target', 'login_recency_category', 'support_risk', 'payment_risk', 'usage_level']


## 1. Define the Prediction Target

The objective is to predict whether a customer will churn.

The `churn_target` column is used as the binary machine learning target:

- 0 = No churn
- 1 = Churn

In [3]:
# Define target variable

target = "churn_target"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (2800, 14)
Target shape: (2800,)

Target distribution:
churn_target
1    1605
0    1195
Name: count, dtype: int64


## 2. Feature Selection

Identifier fields and the original churn label are excluded from the predictive features.

The model should learn churn patterns from customer behavior and subscription characteristics rather than from identifiers or the target itself.

In [4]:
# Remove identifier and target-related columns

X = df.drop(
    columns=[
        "user_id",
        "churn",
        "churn_target"
    ]
)

y = df["churn_target"]

print("Features used for modeling:")
print(X.columns.tolist())

print("\nFeature count:", X.shape[1])

Features used for modeling:
['signup_date', 'plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago', 'login_recency_category', 'support_risk', 'payment_risk', 'usage_level']

Feature count: 12


## 3. Date Feature Preparation

The signup date is converted into a signup year so that the model can capture broad differences between customer signup cohorts without using the raw date value.

In [5]:
# Convert signup date and extract signup year

X["signup_date"] = pd.to_datetime(
    X["signup_date"],
    errors="coerce"
)

X["signup_year"] = X["signup_date"].dt.year

# Remove the original date column
X = X.drop(columns=["signup_date"])

print("Features after date preparation:")
print(X.columns.tolist())

Features after date preparation:
['plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago', 'login_recency_category', 'support_risk', 'payment_risk', 'usage_level', 'signup_year']


## 4. Feature Types

Numerical features are standardized before modeling, while categorical features are one-hot encoded.

This prevents categorical variables from being incorrectly interpreted as numerical quantities.

In [6]:
# Define feature groups

numerical_features = [
    "monthly_fee",
    "avg_weekly_usage_hours",
    "support_tickets",
    "payment_failures",
    "tenure_months",
    "last_login_days_ago",
    "signup_year"
]

categorical_features = [
    "plan_type",
    "login_recency_category",
    "support_risk",
    "payment_risk",
    "usage_level"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Numerical features: 7
Categorical features: 5


## 5. Train-Test Split

The dataset is divided into training and testing sets.

A stratified split is used to preserve the churn/non-churn class distribution in both datasets.

In [7]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training samples: 2240
Testing samples: 560

Training churn distribution:
churn_target
1    0.573
0    0.427
Name: proportion, dtype: float64

Testing churn distribution:
churn_target
1    0.573
0    0.427
Name: proportion, dtype: float64


## 6. Preprocessing Pipeline

Numerical features are standardized using `StandardScaler`, while categorical features are converted into numerical representations using one-hot encoding.

A single scikit-learn pipeline is used to ensure that preprocessing is learned only from the training data.

In [8]:
# Create preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 7. Logistic Regression Baseline

Logistic Regression is used as the baseline classification model.

It provides a simple and interpretable benchmark for predicting customer churn before evaluating more complex machine learning models.

In [9]:
# Create Logistic Regression pipeline

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

print("Logistic Regression pipeline created successfully.")

Logistic Regression pipeline created successfully.


In [10]:
# Train Logistic Regression model

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## 8. Model Predictions

The trained model is used to predict churn outcomes for the unseen test dataset.

Both class predictions and churn probabilities are generated.

In [11]:
# Generate predictions

y_pred = logistic_model.predict(X_test)

y_probability = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))

Predictions generated successfully.
Number of predictions: 560


## 9. Model Evaluation

The baseline model is evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.

Recall is particularly important because failing to identify a customer who is likely to churn can result in a missed retention opportunity.

In [12]:
# Calculate evaluation metrics

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

metrics["Score"] = metrics["Score"].round(4)

metrics

,Metric,Score
0,Accuracy,0.6750
1,Precision,0.7003
2,Recall,0.7570
3,F1-Score,0.7275
4,ROC-AUC,0.7046


In [13]:
print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Retained", "Churned"]
    )
)

Classification Report:
              precision    recall  f1-score   support

    Retained       0.63      0.56      0.60       239
     Churned       0.70      0.76      0.73       321

    accuracy                           0.68       560
   macro avg       0.67      0.66      0.66       560
weighted avg       0.67      0.68      0.67       560



In [14]:
# Generate confusion matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Retained", "Actual Churned"],
    columns=["Predicted Retained", "Predicted Churned"]
)

cm_df

,Predicted Retained,Predicted Churned
Actual Retained,135,104
Actual Churned,78,243


In [15]:
from sklearn.ensemble import RandomForestClassifier

## 10. Random Forest Model

A Random Forest classifier is evaluated as a non-linear alternative to Logistic Regression.

Random Forest can capture interactions and non-linear relationships between customer behavior, engagement, payment activity, and churn.

In [16]:
# Create Random Forest pipeline

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1
            )
        )
    ]
)

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


In [17]:
# Train Random Forest model

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [18]:
# Generate Random Forest predictions

rf_pred = random_forest_model.predict(X_test)

rf_probability = random_forest_model.predict_proba(X_test)[:, 1]

print("Random Forest predictions generated.")

Random Forest predictions generated.


In [19]:
# Evaluate Random Forest

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_probability)

rf_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],
    "Score": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1,
        rf_roc_auc
    ]
})

rf_metrics["Score"] = rf_metrics["Score"].round(4)

rf_metrics

,Metric,Score
0,Accuracy,0.6429
1,Precision,0.6714
2,Recall,0.7383
3,F1-Score,0.7033
4,ROC-AUC,0.7097


In [20]:
print("Random Forest Classification Report:")

print(
    classification_report(
        y_test,
        rf_pred,
        target_names=["Retained", "Churned"]
    )
)

Random Forest Classification Report:
              precision    recall  f1-score   support

    Retained       0.59      0.51      0.55       239
     Churned       0.67      0.74      0.70       321

    accuracy                           0.64       560
   macro avg       0.63      0.63      0.63       560
weighted avg       0.64      0.64      0.64       560



## 11. Model Comparison

The Logistic Regression baseline is compared with Random Forest using multiple evaluation metrics.

In [21]:
# Compare baseline and Random Forest

model_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],
    "Logistic Regression": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ],
    "Random Forest": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1,
        rf_roc_auc
    ]
})

model_comparison.round(4)

,Metric,Logistic Regression,Random Forest
0,Accuracy,0.6750,0.6429
1,Precision,0.7003,0.6714
2,Recall,0.7570,0.7383
3,F1-Score,0.7275,0.7033
4,ROC-AUC,0.7046,0.7097


## 12. Customer Churn Probability

The Logistic Regression model generates a probability score representing the estimated likelihood that each customer will churn.

These probabilities can be used to prioritize customers for retention actions.

In [22]:
# Generate churn probabilities for the test set

churn_probability = logistic_model.predict_proba(X_test)[:, 1]

probability_results = X_test.copy()

probability_results["actual_churn"] = y_test.values
probability_results["churn_probability"] = churn_probability

probability_results["predicted_churn"] = y_pred

probability_results.head()

,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,login_recency_category,support_risk,payment_risk,usage_level,signup_year,actual_churn,churn_probability,predicted_churn
1894,Basic,199,21.0,6,3,32,3,Active,High,High,High,2023,0,0.462641,0
1707,Basic,199,24.9,7,3,31,11,Active,High,High,High,2024,1,0.596594,1
2581,Standard,399,1.7,4,1,1,19,At Risk,Medium,Medium,Low,2024,0,0.704526,1
1450,Basic,199,16.4,7,3,32,26,At Risk,High,High,High,2024,1,0.680900,1
1231,Basic,199,19.9,2,2,35,6,Active,Low,Medium,High,2024,0,0.317605,0


## 13. Customer Risk Classification

Customers are grouped into three risk categories based on predicted churn probability:

- Low Risk: below 40%
- Medium Risk: 40% to below 70%
- High Risk: 70% or above

These thresholds are business segmentation rules and are not model probabilities themselves.

In [23]:
def classify_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"


probability_results["risk_category"] = (
    probability_results["churn_probability"]
    .apply(classify_risk)
)

probability_results["risk_category"].value_counts()

risk_category
Medium Risk    278
High Risk      161
Low Risk       121
Name: count, dtype: int64

In [24]:
high_probability_customers = (
    probability_results
    .sort_values(
        "churn_probability",
        ascending=False
    )
)

high_probability_customers[
    [
        "plan_type",
        "monthly_fee",
        "avg_weekly_usage_hours",
        "support_tickets",
        "payment_failures",
        "tenure_months",
        "last_login_days_ago",
        "actual_churn",
        "churn_probability",
        "risk_category"
    ]
].head(20)

,plan_type,monthly_fee,avg_weekly_usage_hours,support_tickets,payment_failures,tenure_months,last_login_days_ago,actual_churn,churn_probability,risk_category
2460,Basic,199,3.7,8,5,1,45,1,0.958732,High Risk
249,Premium,699,3.6,8,5,23,37,1,0.950733,High Risk
1237,Standard,399,3.8,5,5,22,50,1,0.940444,High Risk
65,Basic,199,1.6,7,4,2,54,1,0.938669,High Risk
2157,Standard,399,4.6,8,4,11,33,1,0.937579,High Risk
1025,Premium,699,0.8,8,3,8,49,1,0.936564,High Risk
530,Premium,699,4.6,5,5,33,41,1,0.935828,High Risk
1269,Standard,399,0.6,6,5,7,53,1,0.930562,High Risk
1033,Premium,699,0.6,8,4,3,24,1,0.922105,High Risk
772,Basic,199,2.7,7,5,8,23,1,0.921880,High Risk


In [25]:
probability_results["risk_category"].value_counts()

risk_category
Medium Risk    278
High Risk      161
Low Risk       121
Name: count, dtype: int64

## 14. Model Persistence

The trained Logistic Regression pipeline is saved so that it can be reused later without retraining the model.

The saved pipeline contains both preprocessing and the trained classifier.

In [26]:
import joblib
from pathlib import Path

# Create model directory if it does not exist
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save trained pipeline
model_path = model_dir / "logistic_regression_churn_model.pkl"

joblib.dump(
    logistic_model,
    model_path
)

print(f"Model saved successfully: {model_path}")

Model saved successfully: ..\models\logistic_regression_churn_model.pkl


## 15. Customer Churn Predictions

The predicted churn probability and risk category are exported for downstream analysis and dashboard development.

In [27]:
# Save customer-level predictions

prediction_output = probability_results.copy()

prediction_path = "../models/customer_churn_predictions.csv"

prediction_output.to_csv(
    prediction_path,
    index=False
)

print(f"Predictions saved successfully: {prediction_path}")

Predictions saved successfully: ../models/customer_churn_predictions.csv
